# GOAPified Plan-and-Execute

This notebook demonstrates how **LangGOAP** replaces the execution routing in
[LangGraph's Plan-and-Execute](https://langchain-ai.github.io/langgraph/tutorials/plan-and-execute/plan-and-execute/)
with formal A\* GOAP planning.

## What GOAP Replaces — and What It Doesn't

The original Plan-and-Execute has two distinct concerns:

1. **Plan generation** — An LLM synthesizes novel subtasks for an unseen goal
   (e.g., "research the hometown of the Australian Open winner" → generate
   search, extract, compose steps).  This requires creativity and world
   knowledge that only an LLM can provide.

2. **Execution routing** — Once a plan exists, the system orders steps,
   executes them, and handles failures.  The original re-prompts the LLM
   to generate a new text plan on failure.

**GOAP replaces concern (2):** given a set of known actions, the A\* planner
discovers the optimal ordering, verifies it is achievable, and the observer
handles failure recovery — all without LLM calls.  Concern (1) — deciding
*what* actions are needed for a novel goal — still requires an LLM or human
to define the action vocabulary.

| Aspect | Original LangGraph | GOAPified (LangGOAP) |
|--------|-------------------|---------------------|
| Action discovery | LLM generates novel subtasks | Predefined action vocabulary |
| Ordering & routing | LLM prompt | A\* formal plan with verified preconditions |
| Failure recovery | LLM re-prompted | Observer detects deviation, A\* replans |
| Plan verification | None | A\* guarantees plan is achievable |
| LLM cost | Per plan + per step | Zero (for planning and routing) |

## Scenario

"What is the hometown of the 2024 Australian Open winner?"

Given three predefined actions (search, extract, compose), the A\* planner
discovers `search_web → extract_facts → compose_response` automatically.

In [1]:
from typing import Any

from langgoap import ActionSpec, GoalSpec, GoapGraph, ReplanStrategy
from tutorial_examples.plan_and_execute import (
    compose_response,
    extract_facts,
    five_step_research_actions,
    plan_and_execute_actions,
    search_web,
)

## Define Plan-and-Execute Actions

Each step in the research pipeline becomes a GOAP action.

In [2]:
# Functions imported from tutorial_examples.plan_and_execute:
#   search_web, extract_facts, compose_response

actions = plan_and_execute_actions()

### GOAP Execution Graph

The planner discovers a plan, the executor runs each action, and the
observer checks progress — replanning automatically if something fails.

In [ ]:
from IPython.display import Image, display

graph = GoapGraph(actions=actions)
display(Image(graph.compile().get_graph().draw_mermaid_png()))

## Formal Plan Discovery

The A\* planner discovers `search_web → extract_facts → compose_response`.
Unlike the original LLM plan, this is **verified** to be achievable.

In [3]:
result = GoapGraph(actions=actions).invoke(
    goal=GoalSpec(conditions={"response_ready": True}),
    world_state={
        "has_task": True,
        "task": "What is the hometown of the 2024 Australian Open winner?",
    },
)

print(f"Status: {result['status']}")
print(f"Response: {result['world_state']['response']}")
print()

# Show the verified action sequence
successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Action sequence: {' → '.join(successful)}")

Status: goal_achieved
Response: The hometown of the 2024 Australian Open winner (Jannik Sinner) is San Candido, South Tyrol, Italy.

Action sequence: search_web → extract_facts → compose_response


In [ ]:
result["plan"].visualize()

## Execution History

Every step produces an `ActionResult` with state snapshots before and after
execution, enabling full traceability.

In [4]:
for entry in result["execution_history"]:
    print(f"  {entry.action_name}: success={entry.success}")
    print(f"    State before keys: {sorted(entry.state_before.keys())}")
    print(f"    State after keys:  {sorted(entry.state_after.keys())}")
    print()

  search_web: success=True
    State before keys: ['has_task', 'task']
    State after keys:  ['has_search_results', 'has_task', 'search_results', 'task']

  extract_facts: success=True
    State before keys: ['has_search_results', 'has_task', 'search_results', 'task']
    State after keys:  ['facts', 'has_extracted_facts', 'has_search_results', 'has_task', 'hometown', 'search_results', 'task', 'winner']

  compose_response: success=True
    State before keys: ['facts', 'has_extracted_facts', 'has_search_results', 'has_task', 'hometown', 'search_results', 'task', 'winner']
    State after keys:  ['facts', 'has_extracted_facts', 'has_search_results', 'has_task', 'hometown', 'response', 'response_ready', 'search_results', 'task', 'winner']



## Replanning on Step Failure

When a step fails (e.g., search API timeout), the observer triggers
replanning from the current state. This replaces the original's LLM
re-prompting with formal replanning.

In [5]:
call_count = {"search": 0}


def flaky_search(ws: dict[str, Any]) -> dict[str, Any]:
    """Fails on first call, succeeds on retry."""
    call_count["search"] += 1
    if call_count["search"] == 1:
        raise RuntimeError("Search API timeout")
    return search_web(ws)


replan_actions = [
    ActionSpec(
        name="search_web",
        preconditions={"has_task": True},
        effects={"has_search_results": True},
        cost=1.0,
        execute=flaky_search,
    ),
    ActionSpec(
        name="extract_facts",
        preconditions={"has_search_results": True},
        effects={"has_extracted_facts": True},
        execute=extract_facts,
    ),
    ActionSpec(
        name="compose_response",
        preconditions={"has_extracted_facts": True},
        effects={"response_ready": True},
        execute=compose_response,
    ),
]

result = GoapGraph(actions=replan_actions).invoke(
    goal=GoalSpec(conditions={"response_ready": True}),
    world_state={"has_task": True},
)

print(f"Status: {result['status']}")
print(f"Replans: {result['replan_count']}")
print(f"Search calls: {call_count['search']}")
failures = [h for h in result["execution_history"] if not h.success]
print(f"Failed attempts: {len(failures)}")

Action 'search_web' failed: Search API timeout


Status: goal_achieved
Replans: 1
Search calls: 2
Failed attempts: 1


## NEVER Replan Strategy

With `ReplanStrategy.NEVER`, the system terminates immediately on failure
instead of attempting recovery.

In [6]:
def always_fails(ws: dict[str, Any]) -> dict[str, Any]:
    raise RuntimeError("permanent failure")


never_actions = [
    ActionSpec(
        name="search_web",
        preconditions={"has_task": True},
        effects={"has_search_results": True},
        execute=always_fails,
    ),
    ActionSpec(
        name="extract_facts",
        preconditions={"has_search_results": True},
        effects={"has_extracted_facts": True},
        execute=extract_facts,
    ),
    ActionSpec(
        name="compose_response",
        preconditions={"has_extracted_facts": True},
        effects={"response_ready": True},
        execute=compose_response,
    ),
]

result = GoapGraph(actions=never_actions).invoke(
    goal=GoalSpec(
        conditions={"response_ready": True},
        replan_strategy=ReplanStrategy.NEVER,
    ),
    world_state={"has_task": True},
)

print(f"Status: {result['status']}")
print(f"Replans: {result['replan_count']}")

Action 'search_web' failed: permanent failure


Status: failed
Replans: 0


## Longer Pipeline: 5-Step Research

The planner handles arbitrarily long pipelines. Here, a 5-step research
pipeline is discovered automatically by A\*.

In [7]:
# Five-step pipeline imported from tutorial_examples.plan_and_execute

result = GoapGraph(actions=five_step_research_actions()).invoke(
    goal=GoalSpec(conditions={"report_complete": True}),
    world_state={"has_task": True, "task": "Research AI planning"},
)

print(f"Status: {result['status']}")
print(f"Report: {result['world_state']['final_report']}")
successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Pipeline: {' -> '.join(successful)}")

Status: goal_achieved
Report: [FINAL] Draft report: Comprehensive analysis of verified facts.
Pipeline: search → verify → analyze → draft → finalize
